# 7.4. Multiple Input and Multiple Output Channels
D2L의 Multiple Input and Multiple Output Channels장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 지금까지 배운 합성곱은 단순화 버전

지금까지 입력과 커널을 2차원 행렬로만 했었다.

```text
2D 입력
   ↓
2D 커널
   ↓
2D Feature Map
```

하지만 실제 컬러 이미지는 2차원이 아니다.

## 2. RGB 이미지의 Channel

일반적인 컬러 이미지는 세 가지 색 정보를 가지고 있다.

예를 들어 아래처럼 세장 행렬로 나눌 수 있다.

```text
R 채널

10  20  30
40  50  60
70  80  90


G 채널

20  40  10
50  70  30
90  20  10


B 채널

30  10  50
20  80  60
10  40  90
```

그래서 RGB 이미지의 shape은 [3, H, W]라고 생각한다. PyTorch에서는 batch까지 [N, C, H, W]를 사용한다.

예를 들어 [32, 3, 224, 224]라면 이미지 32장, RGB 채널 3개, 높이와너비 224이다.

## 3. 입력이 3채널이면 커널도 3채널이어야한다.

입력이 R, G, B라면 커널 하나도 K_R, K_G, K_B 처럼 입력 채널마다 하나씩 가지고 있어야 한다.

예를 들어 kernel size가 3 x 3 이고 RGB 입력이라면 커널 하나의 실제 크기는 [3(C), 3(H), 3(W)] 이다.

## 4. 여러 채널은 어떻게 합성곱할까?

각 채널별로 따로 합성곱한다.

```text
R 입력 × R 커널
        ↓
       결과 R

G 입력 × G 커널
        ↓
       결과 G

B 입력 × B 커널
        ↓
       결과 B

그다음 이 결과를 더한다.

결과 R + 결과 G + 결과 B = 최종 값 하나
```

$$
Y=X_R​∗K_R​+X_G​∗K_G​+X_B​∗K_B​
$$

입력 채널마다 따로 feature map을 만들어서 그대로 출력하는 것이 아니다. 한개의 출력 채널을 만들 때 모든 입력 채널의 계산 결과를 합친다.

## 5. 아주 작은 숫자로 직접 계산해보기

In [ ]:
X = torch.tensor([
    [
        [1., 2.],
        [3., 4.]
    ],
    [
        [5., 6.],
        [7., 8.]
    ]
])

print(X.shape) # 2channels, 2 height, 2 width

torch.Size([2, 2, 2])


In [ ]:
K = torch.tensor([
    [
        [1., 0.],
        [0., 1.]
    ],
    [
        [1., 1.],
        [1., 1.]
    ]
])

print(K.shape) # 커널도 2channels, 2 height, 2 width

첫 번째 채널 = 1x1 + 2x0 + 3x0 + 4x1 = 5
두 번째 채널 = 5x1 + 6x1 + 7x1 + 8x1 = 26

5 + 26 = 31 최종출력은 31 하나이다.

## 6. 그런데 출력 채널은 왜 여러 개일까?

```text
RGB 3채널 입력
     ↓
커널 하나
     ↓
출력 채널 1개
```

하지만 실제 CNN을 보면
```py
nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=3
)
```

커널 세트 64개를 사용하기 때문에 출력이 64개이다.

## 7. 출력 채널 하나마다 별도의 Filter가 있다.

RGB입력에서 출력 채널 하나를 만들기 위해서는

```text
Filter 1

R용 kernel 3×3
G용 kernel 3×3
B용 kernel 3×3
```

이렇게 필요하다. 만약 출력채널 64개를 만들고 싶다면

```text
Filter 1  → Feature Map 1
Filter 2  → Feature Map 2
Filter 3  → Feature Map 3
...
Filter 64 → Feature Map 64
```

보통 하나의 filter = 입력 전체를 보는 kernel 묶음이라고 생각하는 것이 편하다

## 8. Conv2d의 Weight Shape 이해

In [ ]:
conv = nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=3
)

print(conv.weight.shape) # 출력채널, 입력채널, 커널높이, 커널너비

torch.Size([64, 3, 3, 3])


다중 출력 채널을 만들기 위해 출력 채널마다 `Cin × Kh × Kw` 크기의 커널 묶음을 하나씩 갖기 때문에 전체 weight가 이 4차원 구조가 된다.

## 9. 실제 이미지가 통과하면 어떻게 될까?

In [4]:
conv = nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=3,
    padding=1
)

In [ ]:
X = torch.randn(32, 3, 224, 224)

Y = conv(X)

print(X.shape) # [32, 3, 224, 224]
print(Y.shape) # [32, 64, 224, 224]

torch.Size([32, 3, 224, 224])
torch.Size([32, 64, 224, 224])


흐름을 보면

```text
RGB 이미지

[3, 224, 224]

       ↓

Filter 1 ─────────→ Feature Map 1
Filter 2 ─────────→ Feature Map 2
Filter 3 ─────────→ Feature Map 3
...
Filter 64 ────────→ Feature Map 64

       ↓

[64, 224, 224]
```

각 필터 내부에서는 R channel convolution + G channel convolution + B channel convolution 가 수행된다.

## 10. 각 출력 Channel은 무엇을 의미하는가?

직관적으로는 서로 다른 feature를 찾는다고 생각할 수 있다.

예를 들어서
```text
Channel 1 → 세로 경계와 관련된 특징

Channel 2 → 가로 경계와 관련된 특징

Channel 3 → 특정 색 조합

Channel 4 → 곡선

Channel 5 → 질감

...
```

실제 딥러닝에서는 `채널 17번은 눈 검출기다`하고 하나의 의미만 담당한다고 보장되지 않는다. 여러 채널이 함께 유용한 표현을 학습한다.

## 11. CNN이 깊어질수록 채널이 늘어나는 이유

CNN은 보통 이런 형태로 되어있다.

```text
입력
3 × 224 × 224

↓

64 × 224 × 224

↓

128 × 112 × 112

↓

256 × 56 × 56

↓

512 × 28 × 28
```

특징은 이렇다.
```text
H, W
↓
점점 작아짐

Channel
↑
점점 많아짐
```

앞장에서 배운 stride나 pooling으로 공간 크기를 줄이는 대신, 더 많은 채널을 사용해 다양한 feature 표현을 저장하는 것이다.

단순하게 표현하면

```text
초기 Layer

큰 이미지
적은 종류의 feature

↓

깊은 Layer

작은 feature map
많은 종류의 feature
```

이런 식으로 이해하면 된다.

## 12. 1 x 1 Convolution은 왜 쓸까?

In [6]:
nn.Conv2d(
    in_channels=64,
    out_channels=32,
    kernel_size=1
)

Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))

1 x 1이면 주변 픽셀을 안보는데 어떤 의미가 있을까? => 공간을 보는 것이 아니고 Channel을 섞는 것에 의미가 있다.

예를 들어서 한 픽셀 위치에 채널 값이

```text
Channel 1 = 2
Channel 2 = 5
Channel 3 = 1
```

이라면 `1 x 1 Conv`는 이 세 값을 받아서

$$
y=w_1​(2)+w_2​(5)+w_3​(1)+b
$$

처럼 계산한다.

D2L에서는 이것을 각 픽셀 위치마다 동일한 fully connected layer를 적용하는 것과 비슷하게 설명한다. 공간의 H, W는 그대로 두면서 채널 정보를 변환한다.

In [ ]:
conv = nn.Conv2d(
    in_channels=64,
    out_channels=32,
    kernel_size=1
)

X = torch.randn(8, 64, 28, 28) # H, W 는 그대로인데

Y = conv(X)

print(Y.shape) # [8, 32, 28, 28] # 채널만 줄어듦 그래서 1x1 convolution 은 채널 수를 줄이거나 늘리는 용도로 사용한다.

torch.Size([8, 32, 28, 28])


## 13. 핵심 Shape 문제

입력은 [32, 3, 64, 64]

weight와 출력은 어떻게 될까

In [11]:
conv = nn.Conv2d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    padding=1
)

print(conv.weight.shape)
print(conv.weight[0])

torch.Size([16, 3, 3, 3])
tensor([[[ 0.0879, -0.1393, -0.1673],
         [-0.0316, -0.0588,  0.0693],
         [-0.0890,  0.1619,  0.1184]],

        [[-0.0733,  0.0812, -0.0731],
         [-0.0287,  0.1684,  0.1774],
         [-0.0946,  0.0093,  0.0305]],

        [[ 0.1275, -0.0442, -0.1646],
         [-0.0236, -0.1649,  0.1315],
         [ 0.0707,  0.0254, -0.1209]]], grad_fn=<SelectBackward0>)


Weight는 [16, 3, 3, 3] 

16개 output filter, 각 filter마다 3개 input channel x 3 x 3 kernel이 필요하기 때문이다.

출력은 [32, 16, 64, 64] padding = 1이기 때문에 공간 크기는 유지된다.

흐름은 이렇다.
```text
X
[32, 3, 64, 64]

       ↓

Conv Weight
[16, 3, 3, 3]

       ↓

Y
[32, 16, 64, 64]
```

## 14. 오늘의 정리

- 컬러 이미지는 보통 RGB이므로 입력 채널이 3개다.
- RGB 이미지 하나의 shape은 [3, H, W]이다.
- PyTorch에서는 batch까지 포함해서 [N, C, H, W]를 사용한다.
- 입력 채널이 3개라면 하나의 filter도 입력 채널별 kernel을 3개 가지고 있다.
- 각 입력 채널과 해당 kernel을 convolution한 뒤 결과를 모두 더해서 출력 channel 하나를 만든다.
- 출력 channel을 여러 개 만들고 싶다면 filter를 여러 세트 사용한다.
- out_channels=64는 서로 다른 filter 세트 64개를 학습한다는 의미다.
- 하나의 filter는 입력 채널 전체를 동시에 본다.
- 여러 output channel은 서로 다른 feature 표현을 학습할 수 있게 해준다.
- 1×1 Conv는 주변 픽셀보다 channel끼리의 정보를 섞는 연산이라고 이해하면 된다.